# Experiment: FahMai Submission Runner

Objective:
- Run the canonical RAG pipeline from `/Users/chinnphats/Desktop/cedt/Super AI/Competetion/Hack3/fahmai_rag_jina.py` without duplicating logic.
- Generate a candidate submission CSV from the notebook.
- Optionally compare the candidate against the current base `submission.csv` before promoting it.


## Notes

- This notebook intentionally calls the main script instead of re-implementing the pipeline, so notebook behavior stays aligned with the canonical code path.
- By default, output goes to `artifacts/analysis/notebook_submission_candidate.csv` and does **not** overwrite the current base submission.
- Set `question_ids` if you want to run a subset first.


In [ ]:
from __future__ import annotations

import csv
import os
import shlex
import subprocess
from pathlib import Path

PROJECT_ROOT = Path('/Users/chinnphats/Desktop/cedt/Super AI/Competetion/Hack3')
RAG_SCRIPT = PROJECT_ROOT / 'fahmai_rag_jina.py'
BASE_SUBMISSION = PROJECT_ROOT / 'submission.csv'
CANDIDATE_SUBMISSION = PROJECT_ROOT / 'artifacts/analysis/notebook_submission_candidate.csv'
DEBUG_LOG = PROJECT_ROOT / 'artifacts/analysis/notebook_submission_candidate_debug.jsonl'

assert RAG_SCRIPT.exists(), RAG_SCRIPT
PROJECT_ROOT, RAG_SCRIPT


## Configuration

Tune these settings, then rerun the next cells.


In [ ]:
llm_model = 'openthaigpt'
embedding_model = 'jinaai/jina-embeddings-v5-text-small'
retriever = 'hybrid'
reranker_model = 'BAAI/bge-reranker-v2-m3'

# Set to a list like [24, 41, 45] for a smoke test, or None for full 100.
question_ids = None

config = {
    'chunking_strategy': 'markdown',
    'chunk_size': 512,
    'chunk_overlap': 128,
    'top_k': 5,
    'fetch_k': 12,
    'hint_chunks_per_source': 1,
    'max_per_source': 2,
    'candidate_max_per_source': 3,
    'request_timeout': 45,
    'max_retries': 2,
}

flags = {
    'reranker': True,
    'source_aware_retrieval': True,
    'faceted_filtering': True,
    'context_compression': True,
    'deterministic_solvers': True,
    'query_planning': False,
    'choice_verifier': False,
}

config, flags


In [ ]:
def build_command() -> list[str]:
    cmd = [
        'uv', 'run', 'python', str(RAG_SCRIPT),
        '--llm-model', llm_model,
        '--embedding-model', embedding_model,
        '--retriever', retriever,
        '--reranker-model', reranker_model,
        '--chunking-strategy', config['chunking_strategy'],
        '--chunk-size', str(config['chunk_size']),
        '--chunk-overlap', str(config['chunk_overlap']),
        '--top-k', str(config['top_k']),
        '--fetch-k', str(config['fetch_k']),
        '--hint-chunks-per-source', str(config['hint_chunks_per_source']),
        '--max-per-source', str(config['max_per_source']),
        '--candidate-max-per-source', str(config['candidate_max_per_source']),
        '--request-timeout', str(config['request_timeout']),
        '--max-retries', str(config['max_retries']),
        '--debug-log', str(DEBUG_LOG),
        '--output', str(CANDIDATE_SUBMISSION),
    ]

    flag_map = {
        'reranker': '--reranker',
        'source_aware_retrieval': '--source-aware-retrieval',
        'faceted_filtering': '--faceted-filtering',
        'context_compression': '--context-compression',
        'deterministic_solvers': '--deterministic-solvers',
        'query_planning': '--query-planning',
        'choice_verifier': '--choice-verifier',
    }
    for key, flag in flag_map.items():
        cmd.append(flag if flags[key] else '--no' + flag[1:])

    if question_ids:
        cmd.extend(['--ids', ','.join(map(str, question_ids))])
    return cmd

cmd = build_command()
print(' '.join(shlex.quote(part) for part in cmd))


## Run

The next cell runs the canonical script and writes a candidate CSV plus a debug JSONL.


In [ ]:
assert os.getenv('THAILLM_API_KEY') or os.getenv('ThaiLLM'), 'Set THAILLM_API_KEY before running this cell.'

CANDIDATE_SUBMISSION.parent.mkdir(parents=True, exist_ok=True)
DEBUG_LOG.parent.mkdir(parents=True, exist_ok=True)

result = subprocess.run(
    build_command(),
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.stderr:
    print('--- STDERR ---')
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f'Pipeline exited with code {result.returncode}')


## Validate Output

This checks row count and answer range without touching the base submission.


In [ ]:
with CANDIDATE_SUBMISSION.open(encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

row_count = len(rows)
ids = [int(r['id']) for r in rows]
answers = [int(r['answer']) for r in rows]
summary = {
    'row_count': row_count,
    'min_id': min(ids) if ids else None,
    'max_id': max(ids) if ids else None,
    'invalid_answers': [a for a in answers if not (1 <= a <= 10)],
}
summary


## Compare Against Current Base

Use this before promoting a candidate over the current `submission.csv`.


In [ ]:
def load_submission(path: Path) -> dict[int, int]:
    with path.open(encoding='utf-8') as f:
        return {int(r['id']): int(r['answer']) for r in csv.DictReader(f)}

base = load_submission(BASE_SUBMISSION)
cand = load_submission(CANDIDATE_SUBMISSION)
common_ids = sorted(set(base) & set(cand))
diffs = [(qid, cand[qid], base[qid]) for qid in common_ids if cand[qid] != base[qid]]

{
    'match_count': len(common_ids) - len(diffs),
    'total_compared': len(common_ids),
    'diff_count': len(diffs),
    'diffs_preview': diffs[:20],
}


## Promote Candidate Manually

Only do this if the candidate is actually better or matches the trusted base.


In [ ]:
# Uncomment only when you intentionally want to replace the base file.
# BASE_SUBMISSION.write_text(CANDIDATE_SUBMISSION.read_text(encoding='utf-8'), encoding='utf-8')
# print(f'Promoted {CANDIDATE_SUBMISSION} -> {BASE_SUBMISSION}')
